# 🏷️ Comic Studio — Auto Caption Tool

Captions 1300 images with **Florence-2** (natural language, T5-compatible).

## When to run
- Once, before training. Generates `.txt` files next to each image.
- Safe to re-run — already-captioned images are skipped.

## GPU needed
- **None required** (CPU works). T4 (free Colab tier) makes it ~3× faster.

## Steps
Run **A1 → A2 → B1 → C1 → C2** in order.

---
# ━━ SECTION A — Mount Drive ━━

In [ ]:
# A1 — Mount Drive & set paths
from google.colab import drive
drive.mount('/content/drive')
import os

DRIVE_ROOT    = '/content/drive/MyDrive/ComicStudio'
TRAIN_IMAGES  = f'{DRIVE_ROOT}/training_images'   # ← put your 1300 images here
TRIGGER_WORD  = 'comicstyle'                       # ← your trigger word

os.makedirs(TRAIN_IMAGES, exist_ok=True)
imgs = [f for f in os.listdir(TRAIN_IMAGES)
        if f.lower().endswith(('.png','.jpg','.jpeg','.webp'))]
caps = [f for f in os.listdir(TRAIN_IMAGES) if f.endswith('.txt')]
print(f'✅ A1 done')
print(f'   Images found   : {len(imgs)}')
print(f'   Already capped : {len(caps)}')
print(f'   Remaining      : {len(imgs) - len(caps)}')
print(f'   Trigger word   : {TRIGGER_WORD}')
print(f'   Folder         : {TRAIN_IMAGES}')

In [ ]:
# A2 — Quick sanity check: show 3 random images
import random
from IPython.display import display, Image as IPImage
import os

sample = random.sample(imgs, min(3, len(imgs)))
for f in sample:
    path = os.path.join(TRAIN_IMAGES, f)
    display(IPImage(path, width=300))
    txt = path.rsplit('.', 1)[0] + '.txt'
    if os.path.exists(txt):
        print(f'  Caption: {open(txt).read().strip()}')
    else:
        print(f'  Caption: [not yet generated]')

---
# ━━ SECTION B — Install Florence-2 (run once) ━━

In [ ]:
# B1 — Install Florence-2 dependencies (~2 min)
!pip install -q transformers>=4.41 timm einops flash-attn --no-build-isolation
print('✅ B1 done — Florence-2 dependencies installed')

---
# ━━ SECTION C — Generate Captions ━━

In [ ]:
# C1 — Load Florence-2 model
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

FLORENCE_MODEL = 'microsoft/Florence-2-large'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if device == 'cuda' else torch.float32

print(f'Loading Florence-2 on {device}...')
processor = AutoProcessor.from_pretrained(FLORENCE_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    FLORENCE_MODEL, torch_dtype=dtype, trust_remote_code=True
).to(device)
model.eval()
print(f'✅ C1 done — Florence-2 loaded on {device}')

In [ ]:
# C2 — Caption all images  (skips already-captioned ones)
# ═══════════════════════════════════════════════════════
OVERWRITE_EXISTING = False   # set True to re-caption everything
BATCH_LOG_EVERY    = 50      # print progress every N images
# ═══════════════════════════════════════════════════════

import os, time
from PIL import Image
import torch

TASK = '<MORE_DETAILED_CAPTION>'  # Florence-2 task — gives rich descriptions

all_imgs = sorted([
    f for f in os.listdir(TRAIN_IMAGES)
    if f.lower().endswith(('.png','.jpg','.jpeg','.webp'))
])

skipped = done = errors = 0
t0 = time.time()

for i, fname in enumerate(all_imgs, 1):
    img_path = os.path.join(TRAIN_IMAGES, fname)
    txt_path = os.path.splitext(img_path)[0] + '.txt'

    if os.path.exists(txt_path) and not OVERWRITE_EXISTING:
        skipped += 1
        continue

    try:
        img = Image.open(img_path).convert('RGB')
        inputs = processor(text=TASK, images=img, return_tensors='pt').to(device, dtype)
        with torch.no_grad():
            ids = model.generate(
                input_ids=inputs['input_ids'],
                pixel_values=inputs['pixel_values'],
                max_new_tokens=150,
                num_beams=3,
                do_sample=False,
            )
        raw = processor.batch_decode(ids, skip_special_tokens=False)[0]
        caption = processor.post_process_generation(
            raw, task=TASK, image_size=(img.width, img.height)
        )[TASK]
        # Prepend trigger word
        caption = f'{TRIGGER_WORD}, {caption.strip()}'
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(caption)
        done += 1

    except Exception as e:
        print(f'  ⚠️  ERROR on {fname}: {e}')
        errors += 1

    if i % BATCH_LOG_EVERY == 0:
        elapsed = time.time() - t0
        rate = done / elapsed if elapsed > 0 else 0
        remaining = (len(all_imgs) - i) / rate if rate > 0 else 0
        print(f'  [{i}/{len(all_imgs)}] done={done} skip={skipped} err={errors} '
              f'speed={rate:.1f} img/s  ETA={remaining/60:.1f} min')

total_t = time.time() - t0
print(f'\n✅ C2 done in {total_t/60:.1f} min')
print(f'   Captioned : {done}')
print(f'   Skipped   : {skipped}  (already had .txt)')
print(f'   Errors    : {errors}')
print(f'   Speed     : {done/total_t:.2f} img/s')

In [ ]:
# C3 — Review: show 10 random captions for QC
import os, random

txt_files = [f for f in os.listdir(TRAIN_IMAGES) if f.endswith('.txt')]
sample = random.sample(txt_files, min(10, len(txt_files)))
print(f'Showing {len(sample)} random captions:\n')
for f in sample:
    cap = open(os.path.join(TRAIN_IMAGES, f)).read().strip()
    print(f'  {f}')
    print(f'  → {cap}')
    print()

In [ ]:
# C4 — (Optional) Find and fix captions that are too short or missing trigger word
import os

issues = []
for f in os.listdir(TRAIN_IMAGES):
    if not f.endswith('.txt'): continue
    path = os.path.join(TRAIN_IMAGES, f)
    cap = open(path).read().strip()
    if len(cap) < 20:
        issues.append((f, 'too short', cap))
    elif not cap.startswith(TRIGGER_WORD):
        # Auto-fix: prepend trigger word
        fixed = f'{TRIGGER_WORD}, {cap}'
        open(path, 'w').write(fixed)
        issues.append((f, 'trigger added', fixed))

if not issues:
    print('✅ All captions look good!')
else:
    print(f'Found {len(issues)} issues:')
    for fname, reason, cap in issues[:20]:
        print(f'  [{reason}] {fname}: {cap[:80]}')